<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter10/10_01_conical_flow_from_shock_angle.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Taylor-Maccoll Integration from a Prescribed Shock Angle

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 10:** Conical Flow<br>
**Concepts:** conical shock · Taylor-Maccoll equation · Runge-Kutta integration · cone-surface condition

## Learning objectives

- Initialize the conical-flow state immediately behind an oblique shock.
- Integrate the Taylor-Maccoll system from the shock toward the cone.
- Locate the cone surface where the tangential velocity vanishes.

## How to use this notebook

Edit the free-stream Mach number and shock angle in the deterministic example at the bottom of the code cell, then rerun it.

> **Model scope.** The solver assumes steady axisymmetric flow over a sharp cone at zero angle of attack with a calorically perfect gas.


In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional


def fu(a: float, b: float, c: float, gamma: float = 1.4) -> float:
    """
    Governing equation for the second derivative of velocity components
    a = y1 (radial velocity component)
    b = y2 (tangential velocity component)
    c = theta (radians)
    """
    num = (b ** 2) * a - ((gamma - 1.0) / 2.0) * (1.0 - a ** 2 - b ** 2) * (2.0 * a + b / math.tan(c))
    den = ((gamma - 1.0) / 2.0) * (1.0 - a ** 2 - b ** 2) - (b ** 2)
    return num / den


def ft(s: float) -> float:
    """ First derivative of y1 with respect to theta """
    return s


@dataclass
class ConeMachResult:
    beta_deg: float
    beta_rad: float
    theta1_deg: float
    theta1_rad: float
    thetaC_deg: float
    thetaC_rad: float
    Mn1: float
    Mn2: float
    M2: float
    pc: float
    Tc: float
    Roc: float
    iterations: int


def cone_mach_project(
    Minf: float,
    beta_deg: float,
    gamma: float = 1.4,
    h: float = 0.001,
    y2_tol: float = 0.01,
    max_iter: int = 2_000_000,
    store_history: bool = False,
) -> Tuple[ConeMachResult, Optional[Dict[str, List[float]]]]:

    if Minf <= 1.0:
        raise ValueError("Minf must be greater than 1 for supersonic flow.")

    beta = math.radians(beta_deg)

    # Shock existence condition
    beta_min = math.asin(1.0 / Minf)
    if beta <= beta_min:
        raise ValueError(
            f"beta must be greater than asin(1/Minf). asin(1/Minf) = {math.degrees(beta_min):.6f} deg"
        )

    # Theta–beta–Mach relation (oblique shock)
    theta1 = math.atan(
        (2.0 / math.tan(beta))
        * (((Minf * math.sin(beta)) ** 2 - 1.0)
           / (Minf ** 2 * (gamma + math.cos(2.0 * beta)) + 2.0))
    )

    # Normal Mach number upstream of the shock
    Mn1 = Minf * math.sin(beta)

    # Normal Mach number downstream of the shock
    Mn2_sq = (1.0 + ((gamma - 1.0) / 2.0) * (Mn1 ** 2)) / (
        gamma * (Mn1 ** 2) - (gamma - 1.0) / 2.0
    )
    if Mn2_sq <= 0.0:
        raise ValueError("Computed Mn2^2 is non-physical.")

    Mn2 = math.sqrt(Mn2_sq)

    # Downstream Mach number
    sin_term = math.sin(beta - theta1)
    if abs(sin_term) < 1e-12:
        raise ValueError("sin(beta - theta1) is too small.")

    M2 = Mn2 / sin_term

    # Normalized velocity magnitude behind the shock
    Vp = ((2.0 / ((gamma - 1.0) * (M2 ** 2))) + 1.0) ** (-0.5)

    # Initial velocity components
    y1 = Vp * math.cos(beta - theta1)
    y2 = -Vp * math.sin(beta - theta1)

    theta = beta
    dtheta = -abs(h)  # Integration proceeds toward the cone surface

    history = None
    if store_history:
        history = {"theta": [theta], "y1": [y1], "y2": [y2]}

    it = 0
    while it < max_iter:
        it += 1

        # Cone surface condition: tangential velocity vanishes
        if abs(y2) < y2_tol:
            break

        # Runge–Kutta 4th order integration
        L1 = dtheta * ft(y2)
        k1 = dtheta * fu(y1, y2, theta, gamma)

        L2 = dtheta * ft(y2 + k1 / 2.0)
        k2 = dtheta * fu(y1 + L1 / 2.0, y2 + k1 / 2.0, theta + dtheta / 2.0, gamma)

        L3 = dtheta * ft(y2 + k2 / 2.0)
        k3 = dtheta * fu(y1 + L2 / 2.0, y2 + k2 / 2.0, theta + dtheta / 2.0, gamma)

        L4 = dtheta * ft(y2 + k3)
        k4 = dtheta * fu(y1 + L3, y2 + k3, theta + dtheta, gamma)

        y1 += (L1 + 2.0 * L2 + 2.0 * L3 + L4) / 6.0
        y2 += (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0
        theta += dtheta

        if theta <= 1e-10:
            break

        if store_history:
            history["theta"].append(theta)
            history["y1"].append(y1)
            history["y2"].append(y2)

    thetaC = theta

    # Isentropic relations
    Tc = 1.0 + ((gamma - 1.0) / 2.0) * (M2 ** 2)
    pc = Tc ** (gamma / (gamma - 1.0))
    Roc = Tc ** (1.0 / (gamma - 1.0))

    result = ConeMachResult(
        beta_deg=beta_deg,
        beta_rad=beta,
        theta1_deg=math.degrees(theta1),
        theta1_rad=theta1,
        thetaC_deg=math.degrees(thetaC),
        thetaC_rad=thetaC,
        Mn1=Mn1,
        Mn2=Mn2,
        M2=M2,
        pc=pc,
        Tc=Tc,
        Roc=Roc,
        iterations=it,
    )

    return result, history






# Reproducible worked example
result, history = cone_mach_project(
    Minf=2.0,
    beta_deg=40.0,
    h=0.001,
    y2_tol=1e-4,
    store_history=True,
)
print(f"shock angle beta = {result.beta_deg:.4f} deg")
print(f"cone angle theta_c = {result.thetaC_deg:.4f} deg")
print(f"post-shock Mach M2 = {result.M2:.6f}")
print(f"iterations = {result.iterations}")


## Interpretation and verification checklist

- The prescribed shock angle must exceed the Mach angle asin(1/Minf).
- Reduce the angular step and tolerance together to check numerical convergence.
- Confirm that the reported cone angle is smaller than the shock angle.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
